# CRAFT-GC — Full Colab Pipeline

**Before running:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **HF token** (choose one):
   - Colab sidebar **🔑 Secrets** → Add secret named **`HF_TOKEN`** → paste token → **Notebook access ON**
   - Or type when prompted in Cell 1 (hidden input)
3. **Runtime → Run all** (or run cells top to bottom)

| Cell | What | Time |
|------|------|------|
| 1 | Setup + clone repo | ~5 min |
| 2 | Smoke test (2 prompts) | ~5 min |
| 3 | Stage A — CLIP embeddings | ~2 min |
| 4 | Stage B — SD images (50 prompts) | **2–4 hours** |
| 5 | Zip results for download | ~1 min |

Get HF token: https://huggingface.co/settings/tokens (Read access is enough)

In [ ]:
# === CELL 1: SETUP (run first) ===
import os, sys, subprocess, shutil
from getpass import getpass

REPO_URL = "https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git"
PROJECT = "/content/Research"

# Install dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "open-clip-torch", "diffusers", "transformers",
    "accelerate", "pandas", "matplotlib", "seaborn", "huggingface_hub",
], check=True)

# Hugging Face login
from huggingface_hub import login

def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        print("Colab Secret HF_TOKEN not set — enter token below (hidden):")
        return getpass("Hugging Face token: ")

login(token=get_hf_token())

# Safe clone: always chdir to /content BEFORE deleting Research
os.chdir("/content")
if os.path.isdir(PROJECT):
    shutil.rmtree(PROJECT, ignore_errors=True)
subprocess.run(["git", "clone", REPO_URL, PROJECT], check=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT], check=True)

# Verify GPU + package
import torch
import craft_gc

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Set Runtime → T4 GPU → Save → Restart → re-run this cell.")

required = [
    "scripts/colab_run_stage_b.py",
    "scripts/evaluate_embeddings.py",
    "craft_gc/pipeline/craft_gc_pipeline.py",
]
for path in required:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Missing {path} — push latest code to GitHub and re-run Cell 1.")

print("=" * 50)
print("SETUP OK")
print("CWD:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0))
print("craft_gc:", craft_gc.__version__)
print("=" * 50)

In [ ]:
# === CELL 2: SMOKE TEST (2 prompts, ~5 min) — confirms SD + CAS work ===
import subprocess, sys, os

PROJECT = "/content/Research"
os.chdir(PROJECT)

print("Smoke test: 2 prompts × 4 methods × 1 seed = 8 images")
result = subprocess.run(
    [
        sys.executable, "scripts/colab_run_stage_b.py",
        "--device", "cuda",
        "--limit", "2",
        "--seeds", "42",
    ],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError(f"Smoke test failed (exit {result.returncode}). Fix error above before Stage B.")
print("SMOKE TEST PASSED — safe to run Stage B (Cell 4)")

In [ ]:
# === CELL 3: STAGE A — CLIP embedding eval (500 prompts, ~2 min) ===
import os
os.chdir("/content/Research")

!python scripts/evaluate_embeddings.py --device cuda
!python scripts/merge_paper_results.py
!python scripts/plot_figures.py

print("Stage A complete.")

In [ ]:
# === CELL 4: STAGE B — Full SD run (50 prompts, 2–4 HOURS) ===
# DO NOT close this tab. DO NOT restart runtime after this starts.
import subprocess, sys, os

PROJECT = "/content/Research"
os.chdir(PROJECT)

# Pull latest fixes
subprocess.run(["git", "-C", PROJECT, "pull"], check=False)

print("Stage B: 50 prompts × 4 methods × 5 seeds = 1000 images")
print("Expected time: 2–4 hours on T4 GPU")
print("-" * 50)

result = subprocess.run(
    [
        sys.executable, "scripts/colab_run_stage_b.py",
        "--device", "cuda",
        "--limit", "50",
        "--seeds", "42", "123", "456", "789", "1024",
    ],
    cwd=PROJECT,
    capture_output=False,  # show live progress
    text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Stage B failed (exit {result.returncode})")
print("=" * 50)
print("STAGE B COMPLETE")
print("Run Cell 5 to download results.")

In [ ]:
# === CELL 5: Download results as zip ===
import shutil, os
from google.colab import files

os.chdir("/content/Research")

if not os.path.isdir("results/pilot_images"):
    raise FileNotFoundError("No results/pilot_images — run Stage B (Cell 4) first.")

shutil.make_archive("/content/craft_gc_results", "zip", "results")
if os.path.isdir("craft-gc-paper/figures"):
    shutil.make_archive("/content/craft_gc_figures", "zip", "craft-gc-paper/figures")

print("Downloading results zip...")
files.download("/content/craft_gc_results.zip")
if os.path.isfile("/content/craft_gc_figures.zip"):
    print("Downloading figures zip...")
    files.download("/content/craft_gc_figures.zip")

print("Done. On your PC run: python scripts/update_image_table.py")